# Hierarchical tracking of convective structures: example

Test of the `src.tracking` algorithms on 3 timesteps of a MesoNH file.

- **Provided by `src/tracking`**: the generic detector (`detect_objects`), the engine (`HierarchicalTrackingEngine`, `TrackingLevel`) and the tracker (`AdvectedIoUTracker`).
- **Defined here (example)**: the parameters, the detection rules of each level (`examples/convective_rules.py`) and the hierarchy of levels.
- TO RUN THIS NOTEBOOK : MUST ADD A SMALL MESONH SIMULATION

In [ ]:
%matplotlib inline

import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import HTML
from matplotlib.animation import FuncAnimation
from skimage.segmentation import find_boundaries
repo_root = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
sys.path.insert(0, str(repo_root))

from examples.convective_rules import convective_rules
from src.tracking import AdvectedIoUTracker, HierarchicalTrackingEngine, TrackingLevel, detect_objects

## 1. Data (3 timesteps)

In [ ]:
# NEED A MESONH FILE TO RUN, REPLACE <MESONH FILE>

NETCDF_PATH = os.environ.get(
    "NETCDF_PATH",
    "<MESONH FILE>",
)
N_TIMES = 1

ds = xr.open_dataset(NETCDF_PATH)
if "time" in ds.dims:
    frames = [ds.isel(time=i) for i in range(min(N_TIMES, ds.sizes["time"]))]
else:
    frames = [ds]

time_values = [str(f["time"].values).split("T")[-1].split(".")[0] for f in frames]
timestamps_s = [float(pd.Timestamp(f["time"].values).timestamp()) for f in frames]
print(f"{len(frames)} timesteps: {time_values}")

## 2. Parameters and levels

`convective_rules(PARAMS)` turns the variables and thresholds of `PARAMS` into one detection rule per level (the keys of a rule are listed in `src/tracking/detection.py`). The same generic detector, `detect_objects`, is used for all levels.

In [ ]:
PARAMS = {
    "dx_km": 1.0, # horizontal resolution in kilometers
    "min_alt_km": 0.0,
    "max_alt_km": 20.0
}

def convective_rules(params):
    p = params
    grid = {key: p[key] for key in ("dx_km", "min_alt_km", "max_alt_km", "connectivity") if key in p}
    ratio_vars = [ "MRR", "MRI", "MRC", "MRS", "MRG"]

    return {
        "mcs": {
            **grid,
            "conditions": [
                {"variables": ratio_vars, "value": 0.01},
                {"variables": "MSG2_IR108", "op": "<=", "value": 235}, # TB <= 235 K
            ],
            "min_area_km2": 500,
        },
        "core": {
            **grid,
            "conditions": [{"variables": "WT", "value": 4}],
            "split": "WT",  # forced H-maxima segmentation on the updraft speed
            "min_voxels": 5,
            "min_width_px": 3,
            "min_area_px": 5,
        },
        "downdraft" : {
            **grid,
            "conditions": [{"variables": "WT", "value": -2}],
            "split": "WT",  # forced H-maxima segmentation on the downdraft speed
            "min_voxels": 5,
            "min_width_px": 3,
            "min_area_px": 5,
        }
    }

Hierarchy:

- `mcs` is a **root** level;
- `convective_object` is a **child of `mcs`**: its objects are searched within the footprint of an MCS;
- `core` is a **child of `convective_object`**: its cores are searched within a convective object.

Each level has its own tracker (`AdvectedIoUTracker`).

In [ ]:
def tracker(prefix, index_key, **kwargs):
    return AdvectedIoUTracker(track_prefix=prefix, index_key=index_key, dx_meters=PARAMS["dx_km"] * 1000.0, **kwargs)


rules = convective_rules(PARAMS)
levels = [
    TrackingLevel("mcs", detect_objects, parent_name=None, params=rules["mcs"],
                  tracker=tracker("mcs", "indices_2d", min_score=0.01, patience=4, max_speed_ms=30.0)),
    TrackingLevel("core", detect_objects, parent_name="mcs", params=rules["core"],
                  tracker=tracker("core", "indices_3d", min_score=0.1, pre_filter_pixels=30.0)),
    TrackingLevel("downdraft", detect_objects, parent_name="mcs", params=rules["downdraft"],
                  tracker=tracker("downdraft", "indices_3d", min_score=0.1, pre_filter_pixels=30.0)),
]
level_names = [lvl.name for lvl in levels]

## 3. Detection and tracking

In [ ]:
rules = convective_rules(PARAMS)
levels = [
    TrackingLevel("mcs", detect_objects, parent_name=None, params=rules["mcs"],
                  tracker=tracker("mcs", "indices_2d", min_score=0.01, patience=4, max_speed_ms=30.0)),
    TrackingLevel("core", detect_objects, parent_name="mcs", params=rules["core"],
                  tracker=tracker("core", "indices_3d", min_score=0.1, pre_filter_pixels=30.0)),
    TrackingLevel("downdraft", detect_objects, parent_name="mcs", params=rules["downdraft"],
                  tracker=tracker("downdraft", "indices_3d", min_score=0.1, pre_filter_pixels=30.0)),
]
level_names = [lvl.name for lvl in levels]

print(level_names)

engine = HierarchicalTrackingEngine(levels)
result = engine.run_and_track(datasets=frames, time_values=time_values, timestamps_s=timestamps_s)
results, tracks = result["results_by_time"], result["tracks_by_level"]

pd.DataFrame(
    {name: [len(step["levels"][name]) for step in results] for name in level_names},
    index=pd.Index(time_values, name="time"),
)

In [ ]:
from plot_exemples import plot_all_detections
plot_all_detections(0, frames, levels, results, PARAMS)

In [ ]:
from plot_exemples import plot_object_3d

plot_object_3d(0, "core", results, 5)
plot_object_3d(0, "downdraft", results, 5)

## 4. Temporal tracking (animation)

The color of an object is that of its **track**: the same object keeps the same color from one timestep to the next.

In [ ]:
flat_levels = [
    TrackingLevel("mcs", detect_objects, parent_name=None, params=rules["mcs"]),
    TrackingLevel("core", detect_objects, parent_name="mcs", params=rules["core"]),
    TrackingLevel("downdraft", detect_objects, parent_name="mcs", params=rules["downdraft"]),
]
new_flat = HierarchicalTrackingEngine(flat_levels).run(frames, time_values)

# Résultats du moteur de détection
results_by_time = new_flat["results_by_time"]

# Couleurs des niveaux
COLORS = {
    "mcs": "lime",
    "core": "red",
    "downdraft": "blue",
}

# Fenêtre d'affichage : toute l'image
VIEW = np.s_[:, :]


def get_object_coords(obj):
    """Retourne les coordonnées horizontales (y, x) d'un objet."""

    indices_2d = obj.get("indices_2d")
    indices_3d = obj.get("indices_3d")

    if indices_2d is not None and len(indices_2d) > 0:
        return np.asarray(indices_2d, dtype=int)

    if indices_3d is not None and len(indices_3d) > 0:
        # Coordonnées 3D sous la forme (z, y, x)
        return np.asarray(indices_3d, dtype=int)[:, 1:3]

    return np.empty((0, 2), dtype=int)


def plot_frame(t, ax):
    """Affiche les détections d'un instant."""

    ax.clear()

    # Image infrarouge
    tb_da = frames[t]["MSG2_IR108"]

    if tb_da.ndim > 2:
        tb_da = tb_da.isel({
            PARAMS.get("vertical_dim", "level"): 0
        })

    tb_full = np.asarray(tb_da.values).squeeze()

    y_start = VIEW[0].start or 0
    x_start = VIEW[1].start or 0
    tb = tb_full[VIEW]

    ax.imshow(
        tb,
        origin="lower",
        cmap="gray_r",
        interpolation="nearest",
    )

    # Résultats de l'instant courant
    frame_result = results_by_time[t]
    levels_result = frame_result["levels"]

    for level_name, color in COLORS.items():

        if level_name not in levels_result:
            continue

        objects = levels_result[level_name]
        labels = np.zeros(tb.shape, dtype=int)

        for label, obj in enumerate(objects, start=1):
            coords = get_object_coords(obj)

            if len(coords) == 0:
                continue

            # Coordonnées globales vers coordonnées de la fenêtre
            coords = coords.copy()
            coords[:, 0] -= y_start
            coords[:, 1] -= x_start

            valid = (
                (coords[:, 0] >= 0)
                & (coords[:, 0] < tb.shape[0])
                & (coords[:, 1] >= 0)
                & (coords[:, 1] < tb.shape[1])
            )

            coords = coords[valid]

            if len(coords) > 0:
                labels[coords[:, 0], coords[:, 1]] = label

        if labels.max() == 0:
            continue

        # Empreinte des objets
        mask = labels > 0

        overlay = np.ma.masked_where(
            ~mask,
            np.ones(labels.shape),
        )

        ax.imshow(
            overlay,
            origin="lower",
            cmap=ListedColormap([color]),
            alpha=0.35,
            interpolation="nearest",
        )

        # Contours
        boundaries = find_boundaries(labels, mode="inner")

        ax.contour(
            boundaries.astype(float),
            levels=[0.5],
            colors=[color],
            linewidths=0.8,
            origin="lower",
        )

    ax.set_title(f"Détection des objets convectifs — t = {t}")
    ax.set_xlabel("x (pixels)")
    ax.set_ylabel("y (pixels)")


# Création de la figure
fig, ax = plt.subplots(figsize=(9, 8))

legend_handles = [
    Patch(facecolor="lime", edgecolor="lime", alpha=0.5, label="MCS"),
    Patch(facecolor="red", edgecolor="red", alpha=0.5, label="Core"),
    Patch(facecolor="blue", edgecolor="blue", alpha=0.5, label="Downdraft"),
]

ax.legend(handles=legend_handles, loc="upper right")


def update(t):
    plot_frame(t, ax)


# Animation
anim = FuncAnimation(
    fig,
    update,
    frames=min(len(frames), len(results_by_time)),
    interval=300,  # 300 ms par image
    repeat=True,
)

plt.close(fig)

# Affichage dans Jupyter
display(HTML(anim.to_jshtml()))

In [ ]:
ds.close()